Spark's lazy evaluation model is powerful — transformations build a plan, and nothing executes until an action is called. But it also means that every time you call an action on the same DataFrame, Spark re-executes the entire lineage from the source.

Caching breaks that cycle. When you cache a DataFrame, Spark materialises it in memory (or on disk) after the first action, and subsequent actions read from that stored copy instead of rerunning the full plan.

![Caching & Storage Levels](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-caching-storage-levels.svg)

In this notebook you will learn:
- Why repeated actions without caching are expensive
- `cache()` and `persist()` — the difference and when to use each
- Storage levels: `MEMORY_ONLY`, `MEMORY_AND_DISK`, `DISK_ONLY`, and the `_2` variants
- `unpersist()` — releasing cached data when you are done
- Catalog-level caching with `cacheTable` / `uncacheTable`
- Checkpointing — truncating long lineage chains
- How to inspect what Spark currently has cached

### Why Caching Matters

Think of a bank teller who must check a customer's KYC status before every transaction. Without caching, they walk to the file room and retrieve the customer's folder every single time — even for the same customer twice in a row. With caching, they pull the folder out once and keep it on their desk for the rest of the shift.

In Spark terms:
- **Without cache**: every `count()`, `show()`, or `join()` on a DataFrame re-reads the source file and re-applies all transformations
- **With cache**: the first action materialises the result; every subsequent action reads the in-memory copy

Caching pays off when:
1. You call **multiple actions** on the same DataFrame in the same job
2. The DataFrame is used as a **dimension table** in repeated joins
3. You are running **iterative algorithms** (ML training loops) that revisit the same data
4. The computation to produce the DataFrame is expensive (large aggregation, complex filter chain)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, sum as spark_sum
from pyspark import StorageLevel

spark = (
    SparkSession.builder
    .appName("CachingPersistence")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

### Load the Bank Domain Tables

We will use three tables throughout this notebook:
- **`customers`** — small dimension table (CSV), frequently joined with transactions
- **`card_transactions`** — large fact table (Parquet), queried repeatedly for fraud analysis
- **`loan_accounts`** — medium table (JSON multiline), reused for several aggregations

In [ ]:
from pyspark.sql.types import *

customers_schema = StructType([
    StructField("customer_id",   StringType(),  False),
    StructField("full_name",     StringType(),  True),
    StructField("city",          StringType(),  True),
    StructField("credit_score",  IntegerType(), True),
    StructField("kyc_verified",  BooleanType(), True),
])

card_txn_schema = StructType([
    StructField("txn_id",        StringType(),  False),
    StructField("customer_id",   StringType(),  True),
    StructField("card_number",   StringType(),  True),
    StructField("amount",        DoubleType(),  True),
    StructField("merchant",      StringType(),  True),
    StructField("txn_date",      DateType(),    True),
    StructField("is_fraud",      BooleanType(), True),
    StructField("status",        StringType(),  True),
])

loan_schema = StructType([
    StructField("loan_id",       StringType(),  False),
    StructField("customer_id",   StringType(),  True),
    StructField("loan_type",     StringType(),  True),
    StructField("principal",     DoubleType(),  True),
    StructField("interest_rate", DoubleType(),  True),
    StructField("status",        StringType(),  True),
])

customers    = spark.read.csv("data/customers", header=True, schema=customers_schema)
card_txns    = spark.read.parquet("data/card_transactions")
loan_accounts = spark.read.option("multiline", True).json("data/loan_accounts", schema=loan_schema)

print("customers:    ", customers.count())
print("card_txns:    ", card_txns.count())
print("loan_accounts:", loan_accounts.count())

### The Cost of Recomputation (Without Cache)

Below we run two actions — `count()` and `filter().count()` — on `card_txns` without caching it. Each action triggers a full scan of the Parquet files and re-applies all transformations from scratch.

For a DataFrame that only reads a local file this is fast, but imagine the same DataFrame was produced by a complex join across five tables — every action would repeat all of that work.

In [ ]:
import time

# Without cache — each action re-reads and re-processes the Parquet files
t0 = time.time()
total       = card_txns.count()                          # Action 1 — full scan
fraud_total = card_txns.filter(col("is_fraud") == True).count()  # Action 2 — full scan again
t1 = time.time()

print(f"Total transactions : {total:,}")
print(f"Fraud transactions : {fraud_total:,}")
print(f"Elapsed (no cache) : {t1 - t0:.2f}s — two full scans")

### cache() — Materialise and Reuse

`cache()` marks a DataFrame for in-memory storage. It is **lazy** — calling `cache()` alone does nothing. The data is materialised the first time an action is executed after the cache call. All subsequent actions read from the cached copy.

`cache()` is shorthand for `persist(StorageLevel.MEMORY_AND_DISK)` since Spark 3.x — it will spill to disk automatically if the DataFrame is too large to fit entirely in memory.

In [ ]:
# Mark card_txns for caching — nothing is stored yet
card_txns_cached = card_txns.cache()

# First action: materialises the cache
t0 = time.time()
total = card_txns_cached.count()   # Full scan — also writes to cache
t1 = time.time()
print(f"First action (fills cache) : {total:,} rows — {t1 - t0:.2f}s")

# Second action: reads from cache
t0 = time.time()
fraud_total = card_txns_cached.filter(col("is_fraud") == True).count()
t1 = time.time()
print(f"Second action (from cache) : {fraud_total:,} fraud rows — {t1 - t0:.2f}s")

# Third action: also from cache
t0 = time.time()
high_value = card_txns_cached.filter(col("amount") > 10_000).count()
t1 = time.time()
print(f"Third action (from cache)  : {high_value:,} high-value rows — {t1 - t0:.2f}s")

### Inspecting Cache Status

You can check whether a DataFrame is cached and at what storage level using `storageLevel` and `is_cached`.

In [ ]:
print("card_txns_cached.is_cached  :", card_txns_cached.is_cached)
print("card_txns_cached.storageLevel:", card_txns_cached.storageLevel)

print()
print("customers.is_cached          :", customers.is_cached)  # not cached yet

### Storage Levels

`persist()` accepts an explicit `StorageLevel` to control *where* the data is stored.

| StorageLevel | RAM | Disk | Serialized | Replicas | Best For |
|---|---|---|---|---|---|
| `MEMORY_ONLY` | ✓ | — | No | 1 | Small DataFrames that fit in RAM |
| `MEMORY_AND_DISK` | ✓ | ✓ (spill) | No | 1 | Default choice — safe for any size |
| `MEMORY_ONLY_SER` | ✓ | — | Yes | 1 | Large DataFrames — saves RAM via serialization |
| `MEMORY_AND_DISK_SER` | ✓ | ✓ | Yes | 1 | Large DataFrames with disk safety |
| `DISK_ONLY` | — | ✓ | Yes | 1 | Low-memory environments |
| `MEMORY_AND_DISK_2` | ✓ | ✓ | No | 2 | Fast recovery without recompute |
| `OFF_HEAP` | (off-heap) | — | Yes | 1 | Avoid GC pressure (advanced) |

**Rule of thumb**: start with `MEMORY_AND_DISK`. Drop to `MEMORY_ONLY` when you know the data fits and you want the absolute fastest reads. Use `_2` variants only when the cost of recomputing from lineage is very high (e.g., complex multi-hour aggregation).

In [ ]:
from pyspark import StorageLevel

# Persist customers at MEMORY_ONLY — it is small enough to fit entirely in RAM
customers_cached = customers.persist(StorageLevel.MEMORY_ONLY)
customers_cached.count()   # materialise

print("customers storageLevel:", customers_cached.storageLevel)
# StorageLevel(False, True, False, False, 1)
# useDisk=False, useMemory=True, useOffHeap=False, deserialized=False, replication=1

# Persist loan_accounts with MEMORY_AND_DISK (explicit)
loans_cached = loan_accounts.persist(StorageLevel.MEMORY_AND_DISK)
loans_cached.count()  # materialise

print("loans storageLevel    :", loans_cached.storageLevel)

### Caching a Dimension Table for Repeated Joins

The `customers` table is a classic dimension — small (one row per customer), queried often, never changes during a Spark job. Caching it means every join against card_transactions, loan_accounts, or payments reads from memory rather than re-parsing the CSV file each time.

In [ ]:
# customers_cached is already in MEMORY_ONLY from the previous cell

# Join 1: fraud transactions enriched with customer city
fraud_by_city = (
    card_txns_cached
    .filter(col("is_fraud") == True)
    .join(customers_cached, on="customer_id", how="left")
    .groupBy("city")
    .agg(count("*").alias("fraud_count"), spark_sum("amount").alias("fraud_amount"))
    .orderBy(col("fraud_count").desc())
)
fraud_by_city.show(5)

# Join 2: loan accounts enriched with customer credit score
loans_enriched = (
    loans_cached
    .join(customers_cached, on="customer_id", how="left")
    .groupBy("loan_type")
    .agg(avg("credit_score").alias("avg_credit_score"), avg("principal").alias("avg_principal"))
)
loans_enriched.show()
# Both joins read customers_cached from memory — no CSV re-parse

### Caching a Derived / Filtered DataFrame

You can cache not just raw tables but any derived DataFrame — for example, a pre-filtered view of fraud transactions that is used in multiple downstream analyses.

In [ ]:
# Pre-compute and cache the fraud subset — reused in multiple analyses
fraud_txns = (
    card_txns_cached
    .filter(col("is_fraud") == True)
    .select("txn_id", "customer_id", "amount", "merchant", "txn_date")
    .cache()
)

fraud_txns.count()  # materialise

# Analysis 1: fraud by merchant
fraud_txns.groupBy("merchant").agg(count("*").alias("fraud_count")).orderBy(col("fraud_count").desc()).show(5)

# Analysis 2: fraud amount stats — reads from cache, no re-scan
fraud_txns.selectExpr("min(amount) as min_fraud", "max(amount) as max_fraud", "avg(amount) as avg_fraud").show()

### unpersist() — Releasing Cached Data

Cached data stays in memory until one of the following:
- The Spark session ends
- Spark's LRU eviction policy removes it to make room for new data
- You explicitly call `unpersist()`

Always call `unpersist()` once you no longer need the cached DataFrame. This frees up the storage fraction for other DataFrames and avoids unnecessary memory pressure.

`unpersist()` is synchronous by default — it waits until all blocks are actually removed before returning.

In [ ]:
# Done with fraud subset — release its memory
fraud_txns.unpersist()
print("fraud_txns is_cached after unpersist:", fraud_txns.is_cached)

# Release loans as well
loans_cached.unpersist()
print("loans_cached is_cached after unpersist:", loans_cached.is_cached)

### Catalog-Level Caching — cacheTable and uncacheTable

When you register a DataFrame as a temporary view, you can cache the underlying data at the catalog level using `spark.catalog.cacheTable`. This makes the data available to any SQL query or DataFrame operation that reads from that view.

`spark.catalog.isCached` tells you whether a given table name is currently cached.

In [ ]:
# Register customers as a temp view and cache it at the catalog level
customers.createOrReplaceTempView("customers")
spark.catalog.cacheTable("customers")

print("customers cached via catalog:", spark.catalog.isCached("customers"))

# SQL queries against the view now read from the cache
spark.sql("""
    SELECT city, COUNT(*) AS num_customers, AVG(credit_score) AS avg_score
    FROM customers
    GROUP BY city
    ORDER BY avg_score DESC
""").show()

# Uncache when done
spark.catalog.uncacheTable("customers")
print("customers cached after uncache:", spark.catalog.isCached("customers"))

### Checkpointing — Truncating Long Lineage Chains

Caching stores the data in memory but preserves the lineage. Spark can still recompute from the beginning if a cached partition is lost.

**Checkpointing** goes further: it saves the DataFrame to a reliable storage location (HDFS or a local path) and **breaks the lineage**. Spark no longer knows how the data was produced — it just reads the saved files.

When to checkpoint:
- Iterative ML algorithms (e.g., 100-iteration loops) where lineage grows so long that the plan itself becomes a memory and performance problem
- Streaming jobs with stateful operations that accumulate lineage over time
- Any DAG longer than roughly 20–30 shuffle stages

Unlike `cache()`, checkpointing is **eager** — it executes immediately when called.

In [ ]:
import os

# Set a checkpoint directory before checkpointing
checkpoint_dir = "/tmp/spark_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
spark.sparkContext.setCheckpointDir(checkpoint_dir)

# Build a multi-step derived DataFrame
fraud_enriched = (
    card_txns_cached
    .filter(col("is_fraud") == True)
    .join(customers_cached, on="customer_id", how="left")
    .groupBy("city", "status")
    .agg(
        count("*").alias("fraud_count"),
        spark_sum("amount").alias("total_fraud_amount")
    )
)

# Without checkpoint: the plan above is re-evaluated from sources every action
# With checkpoint: saved to disk, lineage broken
fraud_enriched_ckpt = fraud_enriched.checkpoint()  # executes immediately

print("Checkpoint complete — lineage is broken")
fraud_enriched_ckpt.show(5)

### Inspecting the Lineage — Before and After Checkpoint

Use `explain()` to see the effect of checkpointing on the physical plan. Before checkpointing, the plan shows all the source scans and joins. After checkpointing, the plan shows only a simple file scan of the checkpoint directory.

In [ ]:
print("=== BEFORE CHECKPOINT — full lineage ===")
fraud_enriched.explain()

print()
print("=== AFTER CHECKPOINT — lineage truncated ===")
fraud_enriched_ckpt.explain()

### Listing All Cached Tables

Use `spark.catalog.listTables()` to see all registered views and their cache status. You can also navigate to the **Spark UI → Storage tab** to see cached RDD/DataFrame blocks, their sizes, and the fraction of each partition that is cached.

In [ ]:
# Show all registered temp views and whether they are cached
tables = spark.catalog.listTables()
for t in tables:
    cached = spark.catalog.isCached(t.name)
    print(f"  {t.name:<30} tableType={t.tableType:<10} isCached={cached}")

### Anti-Patterns — When NOT to Cache

Caching is not free — it consumes storage memory that would otherwise be available for shuffles, sorts, and joins. Over-caching is a common cause of `OutOfMemoryError` on executors.

**Do not cache when:**

| Scenario | Why it hurts |
|---|---|
| DataFrame used in only one action | Memory consumed but never reused |
| Very large DataFrame that exceeds available RAM | Constant eviction/recompute thrashing |
| DataFrame that changes between actions | Stale cached data produces wrong results |
| Simple scan with no transformations | Source I/O is cheap; cache adds overhead |
| Intermediate step in a long chain that is only used once | Cache the final result instead |

**Cache the right layer**: cache the result of expensive transformations, not raw source reads. If your pipeline is `read → filter → join → aggregate`, cache the post-join result if you need to aggregate it multiple ways — not the raw read.

In [ ]:
# Anti-pattern: caching a DataFrame used in only one action
# This wastes memory — the cached data is never reused
single_use = card_txns_cached.filter(col("amount") > 50_000).cache()
single_use.count()   # one and only action — cache never benefits us
single_use.unpersist()  # always unpersist if you realise you don't need the cache

print("Single-use cache released")

# Better: just call the action directly without cache
count_high = card_txns_cached.filter(col("amount") > 50_000).count()
print(f"High-value transactions (>50k): {count_high:,}")

### Clean Up

Release all remaining cached DataFrames before the session ends.

In [ ]:
card_txns_cached.unpersist()
customers_cached.unpersist()

print("card_txns_cached.is_cached:", card_txns_cached.is_cached)
print("customers_cached.is_cached:", customers_cached.is_cached)

spark.stop()
print("SparkSession stopped")

### Summary

**Why cache**: Spark's lazy evaluation re-executes full lineage for every action. Caching breaks that cycle by materialising a DataFrame after the first action so subsequent actions read from memory.

**`cache()` vs `persist()`**: `cache()` is shorthand for `persist(MEMORY_AND_DISK)`. Use `persist()` with an explicit `StorageLevel` when you need control over serialization, disk spill, or replication.

**Storage levels**:
- `MEMORY_ONLY` — fastest, evicted partitions must be recomputed
- `MEMORY_AND_DISK` — default, safe for any size, spills to disk on eviction
- `DISK_ONLY` — low memory, slowest access
- `_2` variants — replicate to two nodes for instant partition recovery

**When to cache**: dimension tables reused across joins, expensive derived DataFrames used in multiple actions, iterative algorithm inputs.

**`unpersist()`**: always release caches when done — do not rely on LRU eviction to manage memory for you.

**Catalog caching**: `spark.catalog.cacheTable` / `uncacheTable` caches at the SQL view level, visible to all queries against that view.

**Checkpointing**: saves to reliable storage and truncates lineage — use for very long DAGs or iterative algorithms where the plan itself becomes expensive. Unlike `cache()`, checkpointing is eager.